# Imports

Load all the relevant packages

In [ ]:
import dask
import os
import dfm_tools as dfmt
import numpy as np
import xarray as xr
import pandas as pd
import xugrid as xu
import gc

c:\Users\MGE\miniforge3\envs\dfm_py312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from pySVA.sva_core import *
from pySVA.sva_helpers import *

ERROR 1: PROJ: proj_create_from_database: Open of /home/mgeraeds/.conda/envs/dfm_geo_clone1/share/proj failed


# Data description

The pySVA code relies on supplying a data description for the netCDF-dataset that you're loading. For this, you supply a dictionary with the names of the variables in the dataset. The minimum that you need to supply is the cell volume, x-velocity, y-velocity, bed level, vertical viscosity (if no direct vertical diffusivity is present), the depth coordinate, the interfaces coordinate, and, most importantly, the tracer to consider.

If you want to calculate the horizontal dissipation, you should also provide the horizontal diffusivity.

In [ ]:
data_description = {
    "flow_area":"mesh2d_au",
    "volume":"mesh2d_vol1",
    "velx":"mesh2d_ucx",
    "vely":"mesh2d_ucy",
    "velz":"mesh2d_ucz",
    "viscosity":"mesh2d_vicwwu",
    "tracer":"mesh2d_sa1",
    "depth":"mesh2d_flowelem_zcc",
    "interfaces":"mesh2d_flowelem_zw",
    "bed_level":"mesh2d_flowelem_bl",
    "horizontal_diffusivity":"mesh2d_diu"}

# Set up the Dask cluster

We use Dask to process all of the data in parallel. You can run the code on HPC systems, for which you need `dask_jobqueue` and a cluster for HPC-systems (such as the `SLURMCluster`). You can also make use of (threaded) processors in your local machine, for which you use the `LocalCluster`.

You can choose your flavour and run the associated cells below.

## For HPC systems (`SLURMCluster`)

Load the required packages.

In [ ]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client

Choose the number of cores and define the memory limit (which is the total amount of memory available per node divided by the amount of cores per node, times the amount of requested cores).

In [ ]:
n_cores = 64 
mem_lim = str((1440/192)*n_cores) + 'GB' #str(336)+'GB' # str(int(np.floor(336/(n_workers))))+'GB' # 336 for genoa # 224 for rome #n_cores*

Set up the `SLURMCluster`. Take into account that you might need a temporary folder for disk spillage (see https://jobqueue.dask.org/en/stable/generated/dask_jobqueue.SLURMCluster.html for more information).

In [ ]:
cluster = SLURMCluster(name='dask-cluster',
                       cores=n_cores,
                       memory=mem_lim,
                       interface='ib0',
                       queue='fat_genoa',#'genoa',
                       walltime='04:00:00',
                       asynchronous=0)

Scale the cluster (you can increase this depending on the computational power that your computation needs).

In [14]:
cluster.scale(1)

Initialise `Client`.

In [15]:
client = Client(cluster)

Show dashboard link.

In [17]:
client

Connection method: Cluster object,Cluster type: dask_jobqueue.SLURMCluster
Dashboard: http://172.22.59.12:42021/status,
Dashboard: http://172.22.59.12:42021/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://172.22.59.12:38385,Workers: 0
Dashboard: http://172.22.59.12:42021/status,Total threads: 0
Started: Just now,Total memory: 0 B


## For local machines (`LocalCluster`)

Import required packages.

In [3]:
from dask.distributed import LocalCluster, Client

Set up cluster.

In [4]:
cluster = LocalCluster()

Initialise `Client`.

In [ ]:
client = Client(cluster)

The running processes and status of the scheduler can be checked from the dashboard. The next line outputs the dashboard link. Click on it to open the dashboard in your standard browser.

In [6]:
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 7
Total threads: 28,Total memory: 31.72 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:53430,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:53477,Total threads: 4
Dashboard: http://127.0.0.1:53479/status,Memory: 4.53 GiB
Nanny: tcp://127.0.0.1:53433,


# Define  output directory, etc.

In [ ]:
output_dir = ...

# Load dataset using `dfm_tools` Python package.

In [ ]:
file_nc = ... # fill in the path to the file to load

#### **Optional**: define helper function to select a subset in time.

In [22]:
from functools import partial
def _preprocess(x, tstart, tend):
    return x.sel(time=slice(tstart, tend))

In [24]:
upwelling_slice = slice(pd.to_datetime('2019-4-5 00:00'), pd.to_datetime('2019-4-15 00:00'))
upwelling_subslice = slice(pd.to_datetime('2019-4-11 00:00'), pd.to_datetime('2019-4-12 00:00'))

In [ ]:
partial_func = partial(_preprocess, tstart=upwelling_slice.start, tend=upwelling_slice.stop)

In [29]:
data_xr = dfmt.open_partitioned_dataset(file_nc.replace('_0000_',"_*_"), chunks={'time':10}, preprocess=partial_func)

>> xu.open_dataset() with 96 partition(s): 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 : 655.41 sec
>> xu.merge_partitions() with 96 partition(s): 3.54 sec
>> some variables dropped with merging of partitions: ['mesh2d_face_x_bnd', 'mesh2d_face_y_bnd']
>> dfmt.open_partitioned_dataset() total: 658.96 sec


#### Load the data.

In [ ]:
data_xr = dfmt.open_partitioned_dataset(file_nc.replace('_0000_',"_*_"), chunks={'time':10})

#### Make a copy of the dataset.

In [34]:
uds = data_xr.copy()

#### **Optional**: Change the coordinate system of the dataset.

In [31]:
uds.ugrid.set_crs("EPSG:28992")
uds = uds.ugrid.to_crs('WGS84')

Check if new coordinate system is correctly defined.

In [ ]:
gridname = uds.grid.name
uds.ugrid.crs[f'{gridname}'].name == 'WGS 84' # Change to the coordinate system of the data, e.g. 'WGS 84' or 'Amersfoort / RD New'

# Initialize the constructorSVA object.

In [ ]:
# Select the minimum variables needed for salinity variance analysis
uds_new = uds[["mesh2d_au","mesh2d_s1","mesh2d_vol1","mesh2d_ucx","mesh2d_ucy","mesh2d_ucz","mesh2d_vicwwu","mesh2d_sa1","mesh2d_flowelem_zcc","mesh2d_flowelem_zw", "mesh2d_node_z"]]

In [42]:
sva_obj = constructorSVA(uds_new, data_description)

# Calculate terms in the salinity variance budget.

First inspect the data.

In [ ]:
uds_new

Chunk depending on the size of the data.

In [61]:
sva_obj.ds = sva_obj.ds.chunk({"time":5})

Calculate the tracer variance (optional)

In [ ]:
tracer_variance = sva_obj.tracer_variance
tracer_variance = tracer_variance.transpose(*list(sva_obj.tracer.dims)).reset_coords(drop=True)

### **First** calculate the tendency from the not tidally averaged `uds`

In [51]:
tendency = sva_obj.tendency

### **Second**, Calculate the advection, straining, and dissipation.

#### Straining

In [ ]:
straining = sva_obj.straining

In [ ]:
gc.collect()

#### Vertical dissipation

In [ ]:
vert_dissipation = sva_obj.vertical_dissipation

In [ ]:
gc.collect()

#### Horizontal dissipation

In [ ]:
hor_dissipation = sva_obj.horizontal_dissipation

In [ ]:
gc.collect()

#### Advection

In [ ]:
advection = sva_obj.advection

In [ ]:
gc.collect()

# Calculate other terms than from the salinity variance budget.

#### Calculate the divergence.

The divergence is defined as $\left(\frac{\partial u}{\partial x} + \frac{\partial v}{\partial x}\right)$, where $\mathbf{u}_\mathrm{h}=(u,v)$.

In [ ]:
ucx_grad = sva_obj.compute_gradient_on_face(sva_obj.velx).persist()
ucy_grad = sva_obj.compute_gradient_on_face(sva_obj.vely).persist()

In [ ]:
divergence = ucy_grad.isel(mesh2d_nCartesian_coords=1) + ucx_grad.isel(mesh2d_nCartesian_coords=0)

#### Calculate the tracer gradient.

The tracer gradient is defined as $\sqrt{\left(\frac{\partial s}{\partial x}\right)^2 + \left(\frac{\partial s}{\partial y}\right)^2}$, where $s$ is the tracer. This is calculated for each layer. 

In [ ]:
dimn_cart = f'{gridname}_nCartesian_coords'

In [ ]:
gradient = sva_obj.compute_gradient_on_face(sva_obj.tracer)
salinity_gradient = np.abs(np.sqrt(gradient.isel({dimn_cart:0})**2 + gradient.isel({dimn_cart:1})**2)) # in psu/m.

In [ ]:
gc.collect()

#### Calculate the shear

In [ ]:
dudz = differentiate_over_3d_coord(sva_obj.velx, f"{sva_obj.depth.name}", axis=-1)
dudz = dudz.reset_coords(drop=True)

In [ ]:
gc.collect()

In [ ]:
dvdz = differentiate_over_3d_coord(sva_obj.vely, f"{sva_obj.depth.name}", axis=-1)
dvdz = dvdz.reset_coords(drop=True)

In [ ]:
gc.collect()

In [ ]:
shear = np.sqrt(dudz**2 + dvdz**2)